# ERM 424 · Module 2 · First Contact with the Data

Your first look at a dataset nobody has checked yet: classify every column by what it measures, summarize each one the way its type deserves, let a profiling tool take a second look, and then watch what happens when the same data arrives a thousand times larger. Three stages: Stage 1 by hand (about 60 to 75 minutes), Stage 2 with the profiler (about 30 to 40 minutes), Stage 3 at scale (about 30 to 40 minutes, mostly guided).

**Before anything else: File, then Save a copy in Drive.** Work in your copy.

You will submit three things: the completed classification table (Stage 1), your reconciliation findings (Stage 2), and a one-page data-quality note written as a separate document (template at the end). Course objectives served: CO1, CO2, CO5, CO6.

Version 1.2, 2026-11-05. Data dictionary: see the file `m2_claims_DICTIONARY.md` in the course folder for this module.

In [ ]:
# Setup: run this cell first.
import sys, warnings
import numpy as np
import pandas as pd
EXPECTED = {"pandas": "2.2", "numpy": "2.1"}
for name, want in EXPECTED.items():
    have = __import__(name).__version__
    if not have.startswith(want):
        print(f"Note: {name} is {have}; this notebook was built on {want}.x. Most cells will still run.")
%pip install -q fg-data-profiling==4.20.0 duckdb==1.3.2
# Add-on pins verified on the Colab runtime on 2026-10-15 (Python 3.13.16, pandas 2.2.3, numpy 2.1.3); see OPEN_QUESTIONS Q04 and Q10.
import os, time
import matplotlib.pyplot as plt
# Chart colors: the Okabe-Ito palette, readable with most forms of color-vision difference. Meaning never rides on color alone.
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=OKABE_ITO)
plt.rcParams["figure.dpi"] = 100

# Where the course files live. LOCAL = True reads from the course folders next to this notebook;
# LOCAL = False reads from the course repository. At publication, set LOCAL = False and fill in the two URLs.
LOCAL = True   # Q03: no repository yet
BASE_URL = "https://raw.githubusercontent.com/<ORG>/<REPO>/main/data/claims/"
GENERATOR_URL = "https://raw.githubusercontent.com/<ORG>/<REPO>/main/generators/claims_world.py"
DATA_DIR = "../../data/claims"   # notebooks/solutions, notebooks/student and docs/sync sit two folders below the course root
GEN_DIR = "../../generators"
for _cand in ["../../", "../", "./", "/content/"]:   # also works from qa/, from the root, or on a Colab VM with the folders under /content
    if os.path.isdir(_cand + "data/claims"):
        DATA_DIR, GEN_DIR = _cand + "data/claims", _cand + "generators"
        break

def load(name, **kwargs):
    """Read one course CSV from the local data folder (LOCAL) or from the course repository (published)."""
    source = os.path.join(DATA_DIR, name) if LOCAL else BASE_URL + name
    return pd.read_csv(source, **kwargs)

print("pandas", pd.__version__, "| numpy", np.__version__, "| reading data from", os.path.abspath(DATA_DIR) if LOCAL else BASE_URL)


In [ ]:
# Load the Module 2 claims extract and look at the first rows.
df = load("m2_claims.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head()

## Stage 1: Manual

Work through all 18 columns by hand. Classify each one, produce the summaries its type deserves, verify the rule the dictionary states, and record every data-quality issue you find as you go. There are several data-quality issues; they do not all announce themselves the same way.

The six types, in the language of OpenIntro Chapter 1:

- **continuous**: a measurement that can take any value in a range (a temperature, a price).
- **discrete**: a count, whole numbers with meaningful gaps (people in a household, visits per month).
- **ordinal**: categories with a natural order (small, medium, large).
- **nominal**: categories with no order (a color, a country).
- **datetime**: a point on the calendar.
- **identifier**: a label that happens to be a number; arithmetic on it means nothing.

pandas tells you how a column is *stored* (`int64`, `float64`, `object`). It cannot tell you what the column *measures*. That gap is the whole point of this stage.

### Step 1.1: Classify every column

Three cells, in order:

1. Run the first one. It lists every column with the type pandas stores it as, and a `measured_type` that is still blank.
2. In the second one, replace each `""` with one of the six types, written exactly as in the list above (lower case), then run it. Use the data dictionary and what you see in `df.head()`. If a column is a judgment call, pick the best fit and add a short comment at the end of its line saying why, for example `"some_column": "nominal",  # your reason in a few words`.
3. Run the check cell. It tells you if any column is still blank, misspelled, or missing.

In [ ]:
# Classification template: one entry per column, to be filled with one of the six types.
TYPES = ["continuous", "discrete", "ordinal", "nominal", "datetime", "identifier"]
classification = {col: "" for col in df.columns}
pd.DataFrame({"column": df.columns, "storage_type": df.dtypes.astype(str).values,
              "measured_type": [classification[c] for c in df.columns]})

In [ ]:
# Your classification: replace each "" below with one of the six types, then run this cell.
classification.update({
    "claim_id": "",
    "policy_id": "",
    "incident_date": "",
    "report_date": "",
    "claim_type": "",
    "severity": "",
    "vehicles_involved": "",
    "vehicle_make": "",
    "vehicle_year": "",
    "vehicle_value": "",
    "region": "",
    "report_channel": "",
    "adjuster_id": "",
    "injury_amount": "",
    "property_amount": "",
    "vehicle_amount": "",
    "total_amount": "",
    "notes": "",
})
# YOUR CODE HERE

In [ ]:
# Check the classification: every column of df has one of the six types; show measured type next to storage type.
needs_work = [col for col in df.columns if classification.get(col) not in TYPES]
not_columns = [name for name in classification if name not in df.columns]
table = pd.DataFrame({"column": df.columns, "storage_type": df.dtypes.astype(str).values,
                      "measured_type": [classification.get(col, "") for col in df.columns]})
display(table)
if not_columns:
    print(f"{len(not_columns)} name(s) in your classification are not columns of df; check the spelling: {not_columns}")
if needs_work:
    print(f"{len(needs_work)} column(s) still need one of {TYPES}: {needs_work}")
if not needs_work and not not_columns:
    print("All", len(table), "columns classified. This table is deliverable (1).")

### Step 1.2: Summarize every column by its type

The rule: numeric columns get `describe()` and a distribution; categorical columns get `value_counts()`; every column gets a missingness count; identifiers get a uniqueness check, not statistics. Below, each type has one worked pattern on one column. After each pattern, apply it to every other column of that type in the cell marked `# YOUR CODE HERE`. Keep your eyes open while you do it: anything that looks off goes into the issue log in Step 1.4.

In [ ]:
# Pattern, continuous: describe(), a histogram, and the missingness count, shown on total_amount.
col = "total_amount"
print(df[col].describe())
print("missing:", df[col].isna().sum())
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df[col].dropna(), bins=50)
ax.set_title(f"{col}: histogram")
ax.set_xlabel(f"{col} (dollars)")
ax.set_ylabel("number of claims")
plt.show()

Figure: histogram of `total_amount`; nearly every claim sits in the leftmost bars and a handful of very large claims stretch the axis far to the right.

When a histogram is one tall bar and a long flat line, that is information: the column has a long right tail. Module 4 returns to how to draw it well. For now, `describe()` carries the detail the picture hides.

In [ ]:
# Continuous, every column: apply the pattern to every other column you classified as continuous.
# YOUR CODE HERE

In [ ]:
# Pattern, discrete: counts per value in order, a bar chart, and the missingness count, shown on vehicles_involved.
col = "vehicles_involved"
counts = df[col].value_counts().sort_index()
print(counts)
print("missing:", df[col].isna().sum())
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.index.astype(str), counts.values)
ax.set_title(f"{col}: claims per value")
ax.set_xlabel(col)
ax.set_ylabel("number of claims")
plt.show()

Figure: bar chart of claims by number of vehicles involved; two-vehicle events are the most common, then one, with three and four rare.

In [ ]:
# Discrete, every column: apply the pattern to every other column you classified as discrete.
# YOUR CODE HERE

In [ ]:
# Pattern, ordinal: counts in the order the dictionary states, with the share of rows, shown on severity.
col = "severity"
order = ["Minor", "Moderate", "Major", "Total Loss"]
counts = df[col].value_counts().reindex(order)
print(pd.DataFrame({"count": counts, "share": (counts / len(df)).round(3)}))
print("values outside the stated order:", df[col].isin(order).eq(False).sum())
print("missing:", df[col].isna().sum())

In [ ]:
# Ordinal: apply the pattern to any other column you classified as ordinal; then make severity an ordered categorical (pd.Categorical with ordered=True) and use its order once.
# YOUR CODE HERE

In [ ]:
# Pattern, nominal: value counts (missing shown as its own row), distinct count, and the missingness count, shown on claim_type.
col = "claim_type"
print(df[col].value_counts(dropna=False))
print("distinct values:", df[col].nunique())
print("missing:", df[col].isna().sum())
# For a column with many distinct values, print df[col].nunique() and df[col].value_counts().head(10).

In [ ]:
# Nominal, every column: apply the pattern to every other column you classified as nominal.
# YOUR CODE HERE

In [ ]:
# Pattern, datetime: parse the text dates, then range, missingness, and a count per month, shown on incident_date.
col = "incident_date"
dates = pd.to_datetime(df[col], format="%Y-%m-%d")
print("earliest:", dates.min().date(), " latest:", dates.max().date(), " missing:", dates.isna().sum())
per_month = dates.dt.to_period("M").value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(per_month.index.astype(str), per_month.values)
ax.set_title(f"{col}: claims per month")
ax.set_xlabel("month")
ax.set_ylabel("number of claims")
ax.tick_params(axis="x", labelrotation=90)
plt.show()

Figure: bar chart of claims per incident month from January 2024 to June 2025; winter months run a little higher than summer months.

In [ ]:
# Datetime, every column: apply the pattern to every other column you classified as datetime, and describe the lag between the two dates.
# YOUR CODE HERE

In [ ]:
# Pattern, identifier: distinct count, uniqueness, and missingness, shown on claim_id. No describe(): a mean of labels means nothing.
col = "claim_id"
print("rows:", len(df), " distinct:", df[col].nunique(), " unique:", df[col].is_unique, " missing:", df[col].isna().sum())

In [ ]:
# Identifier, every column: apply the pattern to every other column you classified as identifier, and look at how often each value repeats.
# YOUR CODE HERE

In [ ]:
# Missingness for every column at once: count and share of missing values, next to the storage type.
# YOUR CODE HERE

### Step 1.3: Verify the stated rules

The dictionary states two rules that hold in this file:

1. **Consistency rule:** `total_amount` equals `injury_amount` + `property_amount` + `vehicle_amount` on every row.
2. **Date rule:** `report_date` is on or after `incident_date` on every row.

Check both. Compare amounts in cents (multiply by 100 and round) so that floating-point noise does not create false failures, and parse the dates before comparing them. Print the number of rows that fail each rule.

In [ ]:
# Verify the two stated rules: the sum rule in cents, the date rule on parsed dates.
# YOUR CODE HERE

### Step 1.4: Record issues as you go

One row per issue: the column, what you saw, and how you found it. Record; do not fix. Nothing in this notebook changes the data. This log feeds your data-quality note and your discussion post this week.

In [ ]:
# Issue log: one row per issue, recorded as you go. Call note_issue(...) each time you find something.
issues = []

def note_issue(column, what_you_saw, how_you_found_it):
    """Add one row to the issue log and return the log so far."""
    issues.append({"column": column, "what you saw": what_you_saw, "how you found it": how_you_found_it})
    return pd.DataFrame(issues)

# Example of the form, on a column with nothing wrong:
# note_issue("report_date", "every report date is on or after its incident date; no issue", "rule check in Step 1.3")

In [ ]:
# Your issue log entries.
# YOUR CODE HERE

In [ ]:
# Show the issue log as it stands.
pd.DataFrame(issues)

## Stage 2: Profiler

Now let a tool take the second look. The cell below generates a full profiling report on the whole file, saves it as an HTML file you can download from the Colab file browser (the folder icon on the left), and shows it inline. Read the report the way you read your own Stage 1 output: panel by panel, with the dictionary open.

The package is published as `fg-data-profiling` and imported as `data_profiling`. Until 2026 it was called `ydata-profiling`, and before that `pandas-profiling`; older tutorials and videos use those names and the same `ProfileReport` call. Package renames are one reason this notebook pins versions.

Then reconcile. Two things the report surfaced that your manual pass missed, and one thing the report got wrong or mislabeled. Both directions matter: the tool is fast, and it is not audited unless you audit it.

In [ ]:
# Stage 2: generate the full profiling report on the whole file, save it, and show it inline.
from data_profiling import ProfileReport
import data_profiling
print("profiler version:", data_profiling.__version__)
profile = ProfileReport(df, title="First Contact: m2_claims.csv")
profile.to_file("m2_claims_profile.html")
profile.to_notebook_iframe()

In [ ]:
# List the report's alerts as plain text, so you can quote them in your reconciliation.
for alert in profile.description_set.alerts:
    print(alert)
print("\ntype the profiler assigned to each column:")
for col, var in profile.description_set.variables.items():
    print(f"  {col:18s} {var['type']}")

### Reconciliation

**Prompt 1.** Two things the report surfaced that your Stage 1 pass missed (or would have missed without the tool). For each: what the report says, where in the report, and whether you agree after checking it against the data.

> Your answer:

**Prompt 2.** One thing the report got wrong or mislabeled. What it says, what is actually true, and how you know.

> Your answer:

## Stage 3: Scale

Nothing here is graded for correctness; the deliverable is your observation. The cells generate a 2,000,000-row version of the same schema inside your session (no large file is downloaded), then try four things on it: the full profiling report, a minimal report, a full report on a random sample, and one SQL query through DuckDB. Time each one and write down what happened.

In [ ]:
# Stage 3: get the course generator into this session (from the local folder, or downloaded from the course repository).
if LOCAL:
    sys.path.insert(0, os.path.abspath(GEN_DIR))
else:
    import urllib.request
    urllib.request.urlretrieve(GENERATOR_URL, "claims_world.py")
    sys.path.insert(0, ".")
from claims_world import generate_claims, STUDENT_SEED
print("generator loaded; the course seed for this stage is", STUDENT_SEED)

In [ ]:
# Generate the large frame (2,000,000 rows by default) and measure time, size, and peak memory.
N_ROWS = int(os.environ.get("ERM424_QA_ROWS", 2_000_000))   # the QA run can lower this; students leave it alone

def peak_memory_gb():
    """Peak memory of this session so far, in GB (Linux and Colab)."""
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
    except ImportError:
        return float("nan")

t0 = time.perf_counter()
big = generate_claims(N_ROWS, seed=STUDENT_SEED)
generation_seconds = time.perf_counter() - t0
print(f"rows: {len(big):,}   columns: {big.shape[1]}")
print(f"generation time: {generation_seconds:.1f} s")
print(f"in-memory size: {big.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"peak memory of this session so far: {peak_memory_gb():.2f} GB")
big.head()

### The full report, attempted

Run the next cell and watch three things: the elapsed time, the RAM gauge at the top right of Colab, and any "your session crashed" or "running out of memory" message. On a free-tier runtime this cell does not finish in a reasonable time; interrupting it is the expected outcome, not a failure. Give it a few minutes, then Runtime, then Interrupt execution, and write down how long you waited, what the RAM gauge did, and whether the session survived. That observation is the deliverable.

In [ ]:
# Attempt the full profiling report on the large frame. Expect it to run for a long time or exhaust memory; that is the lesson.
ATTEMPT_FULL = (not LOCAL) or os.environ.get("ERM424_ATTEMPT_FULL") == "1"
if ATTEMPT_FULL:
    t0 = time.perf_counter()
    ProfileReport(big, title="Large frame: full report").to_file("big_full_profile.html")
    print(f"full report finished in {time.perf_counter() - t0:.0f} s; peak memory {peak_memory_gb():.2f} GB")
else:
    print("Full-report attempt skipped: LOCAL is True (the local QA run). On Colab this cell runs for real.")
    print("To force it here, set ATTEMPT_FULL = True on the first line of this cell.")

**Prompt 3.** What happened when you attempted the full report: how long you waited, what the RAM gauge did, and how it ended (finished, interrupted, or crashed).

> Your answer:

In [ ]:
# Escape route 1: the minimal report on the whole large frame (no correlations, no interactions, fewer plots).
t0 = time.perf_counter()
ProfileReport(big, title="Large frame: minimal report", minimal=True).to_file("big_minimal_profile.html")
print(f"minimal report on {len(big):,} rows: {time.perf_counter() - t0:.0f} s; peak memory {peak_memory_gb():.2f} GB")

In [ ]:
# Escape route 2: a full report on a random sample of 25,000 rows (seed 424, so everyone draws the same sample).
sample = big.sample(n=25_000, random_state=424)
t0 = time.perf_counter()
ProfileReport(sample, title="Large frame: 25,000-row sample, full report").to_file("big_sample_profile.html")
print(f"full report on {len(sample):,} sampled rows: {time.perf_counter() - t0:.0f} s; peak memory {peak_memory_gb():.2f} GB")

In [ ]:
# Out-of-core query: DuckDB runs SQL directly against the pandas frame, scanning it without copying it into a database.
import duckdb
print("duckdb version:", duckdb.__version__)
t0 = time.perf_counter()
answer = duckdb.query("""
    SELECT claim_type, COUNT(*) AS claims,
           ROUND(SUM(total_amount), 2) AS total_paid, ROUND(AVG(total_amount), 2) AS mean_paid
    FROM big
    GROUP BY claim_type
    ORDER BY total_paid DESC
""").df()
print(f"query over {len(big):,} rows: {time.perf_counter() - t0:.2f} s")
answer.assign(total_paid=answer["total_paid"].map("{:,.2f}".format), mean_paid=answer["mean_paid"].map("{:,.2f}".format))

**Prompt 4.** How did each route behave against the large frame (time, memory, what each one gives up)? One or two sentences per route: minimal report, sampled report, DuckDB query.

> Your answer:

## Data-quality note

Write this as a separate document (one page maximum) and attach it to your submission. Audience: an analyst who has never seen this dataset and has ten minutes. Draft it here if you like, then copy it out.

**What is wrong.** One line per issue, plainest words first.

**Where.** Column, how many rows, how large.

**How you found it.** The check that surfaced each issue; name the ones the profiler missed and the ones it found for you.

**What you recommend.** For each issue: treat, flag, ask the source, or leave; and why.

**At production scale.** One closing sentence: what you would do differently if this dataset arrived with two million rows.

### Before you submit

1. Deliverable (1): the classification table printed by the check cell in Step 1.1, with no column left blank.
2. Deliverable (2): your answers under Prompts 1 and 2 in Stage 2 (Prompts 3 and 4 are your Stage 3 observations; write them, they are not graded for correctness).
3. Deliverable (3): the data-quality note, attached as a document.
4. Your issue log in Step 1.4 shows every issue you found.

## Submit

1. Save your copy to Google Drive if you have not already: File, then Save a copy in Drive.
2. Set the share link so your instructor can view it: Share, then under General access choose Anyone with the link, Viewer.
3. Paste the link into the Canvas assignment.

If anything fails, post in the Q&A discussion or bring it to office hours.